In [ ]:
# Import modules
import paramiko
import os
from dataclasses import dataclass, asdict
import argparse
import json
from datetime import datetime
from pathlib import Path
from typing import Any, Optional
from stat import S_ISDIR, S_ISREG
import pandas as pd
from abc import ABC, abstractmethod
from lxml import etree
import dotenv

In [ ]:
# Provide connection details in separate .env file for security
dotenv.load_dotenv()
hostname = os.getenv('HOSTNAME')  # Replace with your server's IP or hostname
user = os.getenv('USER')    # Replace with your SSH username    
password = os.getenv('PASSWORD')  # SSH password from .env file

PROJECT_DIRECTORY = '/mnt/data_raid/feierabend/AVL_FIRE/PEMWE/PEMStar_2'    # Project directory on the remote server
MODEL_NAME = 'PEMStar_BekaertPTL'                   # AVL FIRE model name
CASE_SET_NAME = 'PolCurve_Bek~rtPTL_Update'        # Case set name within the model
CASE_NAME = None                                    # Set to None to search all cases, or specify a case name like 'Case_1'

# Or, if you want to search for files with specific names
# TARGET_FILENAMES = ['file1.txt', 'report.log']


In [ ]:
# --- SSH Client Initialization ---
ssh_client = paramiko.SSHClient()
# Automatically add the server's host key. For production, it's better to manage known_hosts explicitly.
ssh_client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

try:
    # Connect to the SSH server
    ssh_client.connect(hostname=hostname, username=user, password=password)
    print(f"Connected to {hostname} using password.")

    # Open an SFTP client
    sftp_client = ssh_client.open_sftp()
    print(f"Opened SFTP session.")
except paramiko.AuthenticationException:
    print("Authentication failed. Check your username, password, or private key.")
except paramiko.SSHException as e:
    print(f"Could not establish SSH connection: {e}")  

In [ ]:
# Define function for retrieving remote data files from AVL FIRE model cases
def retrieve_avl_fire_data_paths(sftp_client, project_directory, model_name, case_set_name, data_directory, file_extension, case_name=None):
    simulation_project_path = f"{project_directory}/simulation/project/"
    if case_name is None:
        print("case_name is None, searching for all cases in the specified model and case set...")
        data_paths = []
        for entry in sftp_client.listdir_attr(simulation_project_path):
            if S_ISDIR(entry.st_mode) and f"{model_name}.{case_set_name}." in entry.filename:
                case_path = f"{simulation_project_path}{entry.filename}"
                data_path = f"{case_path}/{data_directory}/{model_name}{file_extension}"
                data_paths.append(data_path)
                print(f"Found case: {entry.filename}, data path: {data_path}")
    else:
        case_set_path = f"{simulation_project_path}{model_name}.{case_set_name}"
        data_path = f"{case_set_path}.{case_name}/{data_directory}/{model_name}{file_extension}"
        print(f"Using specified case_name: {case_name}, data path: {data_path}")
        data_paths = [f"{data_path}"]
    return data_paths
    

# Retrieve Simulation Input Data
Get input from ".asix" file

In [ ]:
input_data_paths = retrieve_avl_fire_data_paths(
    sftp_client=sftp_client,
    project_directory=PROJECT_DIRECTORY,
    model_name=MODEL_NAME,
    case_set_name=CASE_SET_NAME,
    data_directory='input',
    file_extension='.asix'
)


In [ ]:
REMAP_ATTRS = {"t": "type", "v": "value", "u": "unit"}
ATTRS_KEY = "_attrs"

INTERESTING_ATTRS = {
    "type",
    "value",
    "unit",
    "name",
    "index",
    "reference_id",
    "map_type",
    "data_type",
    "orientation",
}


def _local_tag(el: etree._Element) -> str:
    return etree.QName(el).localname


def _try_int(x: Any) -> Optional[int]:
    if x is None:
        return None
    try:
        return int(str(x))
    except Exception:
        return None


def _cast_value(type_str: Optional[str], raw: Any) -> Any:
    if raw is None:
        return None
    if not isinstance(raw, str):
        return raw
    if not type_str:
        return raw

    t = type_str.strip().lower()
    if t == "string":
        return raw
    if t in {"int", "integer"}:
        try:
            return int(raw)
        except ValueError:
            return raw
    if t in {"double", "float", "real"}:
        try:
            return float(raw)
        except ValueError:
            return raw
    if t in {"bool", "boolean"}:
        return raw.strip().lower() in {"yes", "true", "1"}
    if t == "date":
        for fmt in ("%Y%m%d %H:%M:%S", "%Y%m%d"):
            try:
                return datetime.strptime(raw, fmt)
            except ValueError:
                pass
        return raw
    return raw


def _sort_lists_by_index(obj: Any) -> None:
    """Recursively sort any list of dict-nodes by numeric _attrs.index when possible."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == ATTRS_KEY:
                continue
            _sort_lists_by_index(v)
        return

    if isinstance(obj, list):
        idxs = []
        for item in obj:
            if not isinstance(item, dict):
                idxs.append(None)
                continue
            attrs = item.get(ATTRS_KEY, {}) or {}
            idxs.append(_try_int(attrs.get("index")))

        if obj and all(i is not None for i in idxs):
            obj.sort(key=lambda it: _try_int(((it.get(ATTRS_KEY) or {}).get("index"))) or 0)

        for item in obj:
            _sort_lists_by_index(item)


def asix_to_compact_dict(
    root: etree._Element,
    *,
    always_list: bool = False,
    cast_values: bool = False,
    keep_all_attributes: bool = True,
) -> dict[str, Any]:
    """
    Convert ASIX XML root to compact nested dict.

    - always_list=False: singletons are dict; repeats are list
    - always_list=True: children[tag] always list
    - cast_values=True: cast _attrs.value using _attrs.type
    - keep_all_attributes=True: keep all attributes under _attrs (with t/v/u remapped)
    """

    def convert(el: etree._Element, extra_attributes: bool = False) -> dict[str, Any]:
        node: dict[str, Any] = {}

        # Attributes
        attrs_dict: dict[str, Any] = {}
        for k, v in el.attrib.items():
            kk = REMAP_ATTRS.get(k, k)
            if keep_all_attributes or kk in INTERESTING_ATTRS:
                attrs_dict[kk] = v

        if cast_values and "value" in attrs_dict:
            attrs_dict["value"] = _cast_value(attrs_dict.get("type"), attrs_dict.get("value"))

        if attrs_dict:
            if extra_attributes:
                node[ATTRS_KEY] = attrs_dict
            else:
                node.update(attrs_dict)

        # Children grouped by tag
        groups: dict[str, list[dict[str, Any]]] = {}
        for child in el:
            if not isinstance(child.tag, str):  # skip comments/PIs
                continue
            tag = _local_tag(child)
            groups.setdefault(tag, []).append(convert(child))

        for tag, items in groups.items():
            node[tag] = items if always_list else (items[0] if len(items) == 1 else items)

        return node

    out = {_local_tag(root): convert(root)}
    _sort_lists_by_index(out)
    return out


def parse_asix(asix_file, always_list: bool = False, 
               cast_values: bool = False, keep_all_attributes: bool = True) -> dict[str, Any]:
    parser = etree.XMLParser(remove_comments=True, huge_tree=True)
    tree = etree.parse(asix_file, parser)
    root = tree.getroot()
    return asix_to_compact_dict(root, always_list=always_list, 
                                cast_values=cast_values, keep_all_attributes=keep_all_attributes)


def _json_default(o: Any) -> Any:
    if isinstance(o, datetime):
        return o.isoformat()
    raise TypeError(f"Object of type {type(o).__name__} is not JSON serializable")


# with sftp_client.open(data_path, 'r') as data_file:
#     # data = remote_file.read()
#     data_dicts = parse_asix(data_file, always_list=False, keep_all_attributes=True, cast_values=True)

# root_tag = next(iter(data_dicts.keys()))
# top_keys = list((data_dicts[root_tag] or {}).keys())
# print(f"Parsed: str({data_path})")
# print(f"Root tag: {root_tag}")
# print(f"Top-level keys under root (first 30): {top_keys[:30]}")

# with open('testing_output.json', 'w', encoding='utf-8') as f:
#     json.dump(data_dicts, f, default=_json_default, indent=2)

In [ ]:
input_data_dicts_list = []
for data_path in input_data_paths:
    with sftp_client.open(data_path, 'r') as data_file:
        # data = remote_file.read()
        data = parse_asix(data_file, always_list=False, keep_all_attributes=True, cast_values=True)
        input_data_dicts_list.append(data)
    # data_dicts
input_data_dicts_list

In [ ]:
# Close the connections
if 'sftp_client' in locals() and sftp_client:
    sftp_client.close()
    print("SFTP session closed.")
if 'ssh_client' in locals() and ssh_client:
    ssh_client.close()
    print("SSH connection closed.")

In [ ]:
data_dicts = input_data_dicts_list[0]

In [ ]:

for entry in data_dicts:
    # if entry['reference_id'] is not None:
    print(entry)

In [ ]:
with open('testing_output_v3.json', 'w', encoding='utf-8') as f:
    json.dump(data_dicts, f, default=_json_default, indent=2)

# Retrieve Simulation Input Data
Get input from ".asix" file

In [ ]:
results_2d_data_paths = retrieve_avl_fire_data_paths(
    sftp_client=sftp_client,
    project_directory=PROJECT_DIRECTORY,
    model_name=MODEL_NAME,
    case_set_name=CASE_SET_NAME,
    data_directory='results',
    file_extension='.csv'
)
print(results_2d_data_paths)

In [ ]:
data_path = results_2d_data_paths[0]
result_2d_result_list = []
for data_path in results_2d_data_paths:
    with sftp_client.open(data_path, 'r') as data_file:
        df = pd.read_csv(data_file, header=[1,2], sep=';')  # Adjust separator if needed
        result_2d_result_list.append(df)
result_2d_result_list[0].head()

In [ ]:
results_monitoring_data_paths = retrieve_avl_fire_data_paths(
    sftp_client=sftp_client,
    project_directory=PROJECT_DIRECTORY,
    model_name=MODEL_NAME,
    case_set_name=CASE_SET_NAME,
    data_directory='results',
    file_extension='_flc.csv'
)
print(results_monitoring_data_paths)

In [ ]:
data_path = results_monitoring_data_paths[0]
result_monitoring_result_list = []
for data_path in results_monitoring_data_paths:
    with sftp_client.open(data_path, 'r') as data_file:
        df = pd.read_csv(data_file, header=[1,2], sep=';')  # Adjust separator if needed
        result_monitoring_result_list.append(df)
result_monitoring_result_list[0].head()